# Sentence Similarity Search in Embedding Space

This notebook retrieves corpus sentences that are semantically close to predefined modal prompts.

Unlike the prediction notebooks, the input files contain full sentences extracted from the corpus, with columns such as `Reference` and `Sentence`, rather than KWIC concordance lines.

Sentence embeddings are computed with an Italian Sentence-BERT model and indexed with FAISS for nearest-neighbor search.

## Setup

In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas numpy

In [ ]:
import os
import re
import glob
import time

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer
import faiss

## Configuration

In [ ]:
BASE_DIR = "/content"

# File patterns:
# back_lines_*.csv for background (non-modal) files
# modal_sent_*.csv for modal files
BACK_PATTERN = os.path.join(BASE_DIR, "back_lines_*.csv")
MODAL_PATTERN = os.path.join(BASE_DIR, "*sent_*.csv")

METADATA_LINES = 4
SENTENCE_COL_NAME = "Sentence"

MODEL_NAME = "efederici/sentence-BERTino"

NORMALIZE = True
BATCH_SIZE = 256

## Loading full-sentence corpus exports

The corpus used for similarity search combines:

1. randomly sampled background sentences from itTenTen20;
2. targeted sentence exports containing modal constructions.

The input files are Sketch Engine sentence exports.

In [ ]:
def clean_sentence(raw):
    """Clean a raw sentence from SketchEngine export:
    - remove outer quotes
    - remove <s> and </s> tags
    - normalize whitespace
    """
    if pd.isna(raw):
        return None
    s = str(raw).strip()

    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        s = s[1:-1].strip()

    s = s.replace("<s>", " ").replace("</s>", " ")
    s = re.sub(r"\s+", " ", s).strip()

    return s or None


def load_sketchengine_file(path, skip_meta_lines=METADATA_LINES):
    """Load one SketchEngine CSV export:
    - skip metadata lines
    - read header (Reference, Sentence)
    - clean Sentence column
    """
    df = pd.read_csv(path, sep=",", skiprows=skip_meta_lines, engine="python")
    if SENTENCE_COL_NAME not in df.columns:
        raise ValueError(f"Sentence column '{SENTENCE_COL_NAME}' not found in {path}. "
                         f"Columns are: {df.columns.tolist()}")
    df["sentence"] = df[SENTENCE_COL_NAME].apply(clean_sentence)
    df = df.dropna(subset=["sentence"]).copy()
    df["source_file"] = os.path.basename(path)
    return df[["sentence", "source_file"]]


## Loading background sentences

In [ ]:
back_files = sorted(glob.glob(BACK_PATTERN))
print("Found background files:", len(back_files))

back_dfs = []
for f in tqdm(back_files, desc="Loading background files"):
    df = load_sketchengine_file(f)
    back_dfs.append(df)

back = pd.concat(back_dfs, ignore_index=True)
print("Raw background sentences:", len(back))

# Deduplicate by sentence text
back = back.drop_duplicates(subset=["sentence"]).reset_index(drop=True)
print("After deduplication:", len(back))

back.sample(5, random_state=42)

Found background files: 55


Loading background files: 100%|██████████| 55/55 [00:10<00:00,  5.40it/s]


Raw background sentences: 550000
After deduplication: 531651


,sentence,source_file
431661,"Capitare ""per caso"" nella palestra dove si all...",back_lines_5.csv
223916,Tra la fine del Settecento e l''inizio dell''O...,back_lines_3.csv
52481,"- piccolo e compatto: funziona su Windows 95, ...",back_lines_14.csv
218263,Un gruppo di ricerca italiano ha scoperto l''e...,back_lines_3.csv
344212,La tradizionale musica africana ha segnato l''...,back_lines_41.csv


## Loading modal sentences

In [ ]:
modal_files = sorted(glob.glob(MODAL_PATTERN))
print("Found modal files:", len(modal_files))

modal_dfs = []
for f in tqdm(modal_files, desc="Loading modal files"):
    df = load_sketchengine_file(f)
    modal_dfs.append(df)

modal = pd.concat(modal_dfs, ignore_index=True)
print("Raw modal sentences:", len(modal))

# Deduplicate
modal_clean = modal.drop_duplicates(subset=["sentence"]).reset_index(drop=True)
print("After deduplication:", len(modal_clean))

modal_clean.sample(5, random_state=42)

Found modal files: 6


Loading modal files: 100%|██████████| 6/6 [00:01<00:00,  3.83it/s]

Raw modal sentences: 60000
After deduplication: 59907


,sentence,source_file
37795,La maggior parte degli sport acquatici si poss...,potere_sent_2.csv
49677,Adesso volevo andare a cercare qualche canzone...,volere_sent_1.csv
1180,"E'' per questo che noi dobbiamo essere , a mio...",dovere_sent_1.csv
18276,"Trovarsi a dover girare su sé stessi, sempre n...",dovere_sent_2.csv
53883,@axel ... a parte che anche una foto di un mur...,volere_sent_2.csv


## Building the searchable sentence corpus

In [ ]:
back["is_modal"] = 0
modal_clean["is_modal"] = 1

all_corpus = pd.concat([back, modal_clean], ignore_index=True)
all_corpus = all_corpus.drop_duplicates(subset=["sentence"]).reset_index(drop=True)

n_total = len(all_corpus)
n_modal = int(all_corpus["is_modal"].sum())
print("Final corpus size:", n_total)
print("Modal sentences:", n_modal)
print("Modal share: {:.2f}%".format(100 * n_modal / n_total))

all_corpus.head()

Final corpus size: 591558
Modal sentences: 59907
Modal share: 10.13%


,sentence,source_file,is_modal
0,"Cari Amici Lettori de L''ARENA DI POLA, cari S...",back_lines_1.csv,0
1,Qualche anticipazione: - Il redazionale del pr...,back_lines_1.csv,0
2,Amichevole scambio di ospitalità musicale fra ...,back_lines_1.csv,0
3,"Abbiamo chiuso anche il numero di agosto, mese...",back_lines_1.csv,0
4,"Ma non solo, perché le tematiche che seguiamo ...",back_lines_1.csv,0


## Sentence embedding

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device.upper()}")

model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 128

def encode_in_batches(sentences, batch_size=BATCH_SIZE):
     """Encode sentences into a NumPy array of embeddings."""
    start_time = time.time()
    embeddings = model.encode(
        sentences,
        batch_size=batch_size,
        show_progress_bar=True,     # tqdm built-in progress bar
        convert_to_numpy=True,
        normalize_embeddings=False  # manual normalization below
    )
    elapsed = time.time() - start_time
    print(f"Encoded {len(sentences):,} sentences in {elapsed/60:.2f} minutes.")
    return embeddings

sentences = all_corpus["sentence"].tolist()
embeddings = encode_in_batches(sentences)
print("Embeddings shape (before normalization):", embeddings.shape)

# L2 normalization (so that cosine similarity = inner product)
if NORMALIZE:
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    embeddings = embeddings / norms

embeddings = embeddings.astype(np.float32, copy=False)

emb_dim = embeddings.shape[1]
print(f"Embedding dimension: {emb_dim}")
print(f"Embeddings dtype: {embeddings.dtype}")

## FAISS indexing

In [ ]:
index = faiss.IndexFlatIP(emb_dim)
index.add(embeddings)
print("FAISS index size (number of vectors):", index.ntotal)

FAISS index size (number of vectors): 591558


## Query prompts

The following prompt sets are used to construct query embeddings. Each query embedding is obtained by averaging the sentence embeddings of its prompt sentences.

Prompt sets include:
- general modal constructions;
- modal-specific constructions;
- examples associated with epistemic, deontic, and dynamic readings.

In [ ]:
def average_embedding(prompt_sentences):
    """Compute a normalized average embedding for a list of prompt sentences."""
    start_time = time.time()
    vecs = model.encode(
        prompt_sentences,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )

    if NORMALIZE:
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        vecs = vecs / norms

    avg = vecs.mean(axis=0, keepdims=True)

    if NORMALIZE:
        norm = np.linalg.norm(avg, axis=1, keepdims=True)
        norm[norm == 0] = 1.0
        avg = avg / norm

    avg = avg.astype(np.float32, copy=False)

    elapsed = time.time() - start_time
    print(f"Averaged {len(prompt_sentences)} prompts in {elapsed:.1f}s")
    return avg

In [ ]:
# General modal+INF construction prompts
PROMPTS_ALL = [
    "Devo finire il lavoro.",
    "Devi consegnare il modulo.",
    "Deve essere vero.",
    "Non si deve entrare.",
    "Posso aiutarti a risolvere il problema.",
    "Potrebbe essere tardi",
    "Voi potete venire con noi.",
    "Loro possono accedere.",
    "Vorrei vedere il film.",
    "Vogliamo discutere con te.",
    "Non vogliono rinunciare all'idea.",
    "Voglio andare a casa subito.",
]

# Per-modal
PROMPTS_DOVERE = [
    "Devo finire il lavoro.",
    "Deve aver commesso un errore.",
    "Devi arrivare puntuale.",
    "Lui deve consegnare i documenti.",
    "Dobbiamo rispettare le regole.",
    "Non devo dimenticare l'appuntamento.",
    "Si deve entrare subito.",
    "Per uscire si deve passare di lì.",
    "Prima o poi doveva accadere.",
    "Dovrei averlo già mandato.",
    "La strada è chiusa, si deve deviare.",
    "Avreste dovuto ascoltare l'avviso.",
]

PROMPTS_POTERE = [
    "Posso risolvere il problema.",
    "Puoi sederti qui.",
    "Lui può partecipare alla gara.",
    "Potete uscire più tardi.",
    "Non posso rispondere adesso.",
    "Io posso fare la verticale.",
    "Tutto può finire.",
    "Potrebbe essere troppo tardi.",
    "Può aver detto qualcosa.",
    "Qui non si può parcheggiare.",
    "Possiamo correre velocemente.",
    "Possono venire con noi?",
]

PROMPTS_VOLERE = [
    "Vuoi iniziare il progetto.",
    "Voglio partire domani.",
    "Vorrei aggiungere un commento.",
    "Vogliamo sapere cosa pensi.",
    "Vorremmo discutere con te.",
    "Non volevano aspettare ancora.",
    "Lo voglio condividere con lui.",
    "Avrei voluto dire una cosa.",
    "Vorremmo ringraziarti per la collaborazione.",
    "Loro vogliono sottolineare un punto.",
    "La procedura vuole seguire questi passaggi.",
    "La legge vuole tutelare i cittadini.",
]

# Per-reading
PROMPTS_EPISTEMIC = [
    "Deve essere tardi.",
    "Deve essere a casa ormai.",
    "Può essere già arrivato.",
    "Potrebbe essere stato lui.",
    "Devono aver dimenticato le chiavi.",
    "Deve aver detto qualcosa.",
    "Può essere un errore.",
    "Potrebbe aver fatto un dolce.",
    "Deve essere successo ieri.",
    "Potrebbe aver cambiato idea.",
    "Dovrebbe essere di Marco.",
    "Può essere già finita.",
]

PROMPTS_DEONTIC = [
    "Devi pagare la multa.",
    "Si deve indossare il casco.",
    "Non dovete superare il limite.",
    "Devi fare quello che dico.",
    "Non si può fumare.",
    "Puoi esimerti dal compito.",
    "Qui non si può entrare.",
    "Non potete avvicinarvi alla linea.",
    "La direttiva vuole applicare la misura.",
    "La tradizione vuole osservare questa regola.",
    "Il contratto vuole garantire la sicurezza.",
    "Voglio tornare subito a casa.",
]

PROMPTS_DYNAMIC = [
    "Io posso correre per dieci chilometri.",
    "Benedetta può nuotare velocemente.",
    "Possono suonare il pianoforte.",
    "Dobbiamo riposare per andare avanti.",
    "Qui è libero, puoi parcheggiare.",
    "Per uscire devi girare.",
    "Per attraversare devi passare di lì.",
    "La porta è bloccata, non puoi accedere.",
    "Può piovere d'inverno.",
    "Tutte le navi possono affondare.",
    "Deve finire prima o poi.",
    "Devo mangiare o crollo.",
]


## Building query embeddings

In [ ]:
queries = {
    "all_modal":       average_embedding(PROMPTS_ALL),
    "dovere":          average_embedding(PROMPTS_DOVERE),
    "potere":          average_embedding(PROMPTS_POTERE),
    "volere":          average_embedding(PROMPTS_VOLERE),
    "epistemic":       average_embedding(PROMPTS_EPISTEMIC),
    "deontic":         average_embedding(PROMPTS_DEONTIC),
    "dynamic":         average_embedding(PROMPTS_DYNAMIC),
}

list(queries.keys())

## Nearest-neighbor retrieval

In [ ]:
def retrieve_neighbors(query_vec, top_k=100):
    """Retrieve top_k nearest neighbors for a given query vector."""
    scores, idx = index.search(query_vec, top_k)
    idx = idx[0]
    scores = scores[0]
    out = all_corpus.iloc[idx].copy()
    out["score"] = scores
    return out


In [ ]:
TOP_K = 100

neighbors_all_modal = retrieve_neighbors(queries["all_modal"], TOP_K)
neighbors_dovere    = retrieve_neighbors(queries["dovere"], TOP_K)
neighbors_potere    = retrieve_neighbors(queries["potere"], TOP_K)
neighbors_volere    = retrieve_neighbors(queries["volere"], TOP_K)

neighbors_epistemic = retrieve_neighbors(queries["epistemic"], TOP_K)
neighbors_deontic   = retrieve_neighbors(queries["deontic"], TOP_K)
neighbors_dynamic   = retrieve_neighbors(queries["dynamic"], TOP_K)

neighbors_all_modal.head(10)

,sentence,source_file,is_modal,score
541206,Come devo fare .,dovere_sent_1.csv,1,0.687075
155531,Non cosa fare.,back_lines_23.csv,0,0.678291
331979,È possibile.,back_lines_4.csv,0,0.664908
167968,Ma non so come fare.,back_lines_25.csv,0,0.653818
265036,Basta chiedere.,back_lines_33.csv,0,0.646672
551544,Non devi chiederti perchè.,dovere_sent_2.csv,1,0.642205
573082,Che ci vuoi fare ).,volere_sent_1.csv,1,0.641054
81275,Non so cosa fare.,back_lines_17.csv,0,0.640570
327821,O non fare.,back_lines_4.csv,0,0.634784
262563,Non so cosa fare...,back_lines_33.csv,0,0.634200


In [ ]:
OUT_DIR = os.path.join(BASE_DIR, "similarity_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

neighbors_all_modal.to_csv(os.path.join(OUT_DIR, "neighbors_all_modal.csv"), index=False)
neighbors_dovere.to_csv(os.path.join(OUT_DIR, "neighbors_dovere.csv"), index=False)
neighbors_potere.to_csv(os.path.join(OUT_DIR, "neighbors_potere.csv"), index=False)
neighbors_volere.to_csv(os.path.join(OUT_DIR, "neighbors_volere.csv"), index=False)

neighbors_epistemic.to_csv(os.path.join(OUT_DIR, "neighbors_epistemic.csv"), index=False)
neighbors_deontic.to_csv(os.path.join(OUT_DIR, "neighbors_deontic.csv"), index=False)
neighbors_dynamic.to_csv(os.path.join(OUT_DIR, "neighbors_dynamic.csv"), index=False)

OUT_DIR
